# Phase 2: Physics-Informed Neural Network (PINN)
Welcome to the Brain of the engine. Here, we will train a neural network to approximate the 3D Heat Equation.

### The Goal
Instead of calculating `T_new = T_old + (alpha * dt * laplacian)` over a massive voxel grid, we will train an AI so that we can ask it: `Predict_Temp(x, y, z, time)` and get an instant, continuous answer.

In [16]:
import sys
# Scrub the corrupted local installation
!{sys.executable} -m pip uninstall -y torch torchvision torchaudio
!{sys.executable} -m pip install --no-cache-dir torch --index-url https://download.pytorch.org/whl/cpu

Found existing installation: torch 2.12.0+cpu
Uninstalling torch-2.12.0+cpu:
  Successfully uninstalled torch-2.12.0+cpu


Looking in indexes: https://download.pytorch.org/whl/cpu
   ---------------------------------------- 0.0/122.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/122.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/122.9 MB ? eta -:--:--
   ---------------------------------------- 1.0/122.9 MB 2.4 MB/s eta 0:00:51
    --------------------------------------- 1.8/122.9 MB 3.1 MB/s eta 0:00:40
    --------------------------------------- 2.6/122.9 MB 3.3 MB/s eta 0:00:37
   - -------------------------------------- 3.4/122.9 MB 3.3 MB/s eta 0:00:37
   - -------------------------------------- 3.9/122.9 MB 3.3 MB/s eta 0:00:36
   - -------------------------------------- 3.9/122.9 MB 3.3 MB/s eta 0:00:36
   - -------------------------------------- 4.2/122.9 MB 2.6 MB/s eta 0:00:47
   - -------------------------------------- 4.7/122.9 MB 2.5 MB/s eta 0:00:47
   - -------------------------------------- 5.0/122.9 MB 2.4 MB/s eta 0:00:49
   - -----------------------

In [17]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Check if Google Colab GPU (T4) or local GPU is active
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch Version: {torch.__version__}")
print(f"Using Device: {device}")

PyTorch Version: 2.12.0+cpu
Using Device: cpu


In [18]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch Version: {torch.__version__}")
print(f"Using Device: {device}")

# --- 1. CORE CONSTANTS ---
L = 0.1                # Shield thickness (m) — domain is [0,L] in x, y, z
T_INITIAL = 300.0      # Room temperature (K)
ALPHA = 0.0000068      # Thermal diffusivity of Ti-6Al-4V (m²/s)
K_TI = 21.9            # Thermal conductivity (W/m·K)
FLUX_TARGET = 500000.0 # Plasma heat flux (W/m²) — applied to z=0 face
T_SCALE = 2000.0       # Normalization scale for temperature

# --- 2. THE 3D PHYSICS LOSS ---
def Calculate_loss_3D(model, x, y, z, t, alpha):
    x.requires_grad_(True)
    y.requires_grad_(True)
    z.requires_grad_(True)
    t.requires_grad_(True)
    T = model(x, y, z, t)
    
    # dT/dt
    dT_dt = torch.autograd.grad(T, t, torch.ones_like(T), create_graph=True)[0]
    
    # d²T/dx²
    dT_dx = torch.autograd.grad(T, x, torch.ones_like(T), create_graph=True)[0]
    d2T_dx2 = torch.autograd.grad(dT_dx, x, torch.ones_like(dT_dx), create_graph=True)[0]
    
    # d²T/dy²
    dT_dy = torch.autograd.grad(T, y, torch.ones_like(T), create_graph=True)[0]
    d2T_dy2 = torch.autograd.grad(dT_dy, y, torch.ones_like(dT_dy), create_graph=True)[0]
    
    # d²T/dz²
    dT_dz = torch.autograd.grad(T, z, torch.ones_like(T), create_graph=True)[0]
    d2T_dz2 = torch.autograd.grad(dT_dz, z, torch.ones_like(dT_dz), create_graph=True)[0]
    
    # 3D Heat Equation: dT/dt - alpha * (d²T/dx² + d²T/dy² + d²T/dz²) = 0
    residue = dT_dt - alpha * (d2T_dx2 + d2T_dy2 + d2T_dz2)
    return torch.mean((residue / T_SCALE)**2)

# --- 3. HARD-CONSTRAINED 3D ARCHITECTURE ---
class MetShield3D(nn.Module):
    def __init__(self):
        super().__init__()
        # Wider network: 4 inputs (x,y,z,t) -> 256 hidden neurons
        self.net = nn.Sequential(
            nn.Linear(4, 256), nn.Tanh(),
            nn.Linear(256, 256), nn.Tanh(),
            nn.Linear(256, 256), nn.Tanh(),
            nn.Linear(256, 256), nn.Tanh(),
            nn.Linear(256, 1)
        )

    def forward(self, x, y, z, t):
        # Coordinate stretching for all 3 spatial axes
        x_s = x / L
        y_s = y / L
        z_s = z / L
        raw_output = self.net(torch.cat([x_s, y_s, z_s, t], dim=1))
        
        # Hard initial condition: T = T_INITIAL at t=0
        return T_INITIAL + (t * T_SCALE * raw_output)

model = MetShield3D().to(device)
print(model)


PyTorch Version: 2.12.0+cpu
Using Device: cpu
MetShield3D(
  (net): Sequential(
    (0): Linear(in_features=4, out_features=256, bias=True)
    (1): Tanh()
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): Tanh()
    (4): Linear(in_features=256, out_features=256, bias=True)
    (5): Tanh()
    (6): Linear(in_features=256, out_features=256, bias=True)
    (7): Tanh()
    (8): Linear(in_features=256, out_features=1, bias=True)
  )
)


In [ ]:
# --- TRAINING CONSTANTS ---
N_PDE = 15000     # Collocation points inside the 3D domain
N_BC = 500        # Points per boundary face
ADAM_EPOCHS = 3000
LBFGS_MAX_ITER = 2500

# Helper: generate random points inside [0, L]^3 x [0, 1]
def sample_domain(n):
    x = (torch.rand(n, 1, device=device) * L).requires_grad_(True)
    y = (torch.rand(n, 1, device=device) * L).requires_grad_(True)
    z = (torch.rand(n, 1, device=device) * L).requires_grad_(True)
    t = torch.rand(n, 1, device=device).requires_grad_(True)
    return x, y, z, t

# Helper: sample random coords for 2 free axes + time, within [0, L] and [0, 1]
def sample_face(n):
    a = torch.rand(n, 1, device=device) * L
    b = torch.rand(n, 1, device=device) * L
    t = torch.rand(n, 1, device=device)
    return a, b, t

def compute_all_losses(model):
    # ---- PDE Loss ----
    x_p, y_p, z_p, t_p = sample_domain(N_PDE)
    loss_pde = Calculate_loss_3D(model, x_p, y_p, z_p, t_p, ALPHA)
    
    # ---- FACE 1: Front (z=0) — Neumann flux boundary ----
    a, b, t_f = sample_face(N_BC)
    z_front = torch.zeros(N_BC, 1, device=device, requires_grad=True)
    pred_front = model(a, b, z_front, t_f)
    dT_dz_front = torch.autograd.grad(pred_front, z_front, torch.ones_like(pred_front), create_graph=True)[0]
    # -K * dT/dz = FLUX  =>  dT/dz = -FLUX/K
    loss_front = torch.mean(((-K_TI * dT_dz_front - FLUX_TARGET) / FLUX_TARGET)**2)
    
    # ---- FACE 2: Back (z=L) — Dirichlet, T = T_INITIAL ----
    a2, b2, t_b = sample_face(N_BC)
    z_back = torch.ones(N_BC, 1, device=device) * L
    pred_back = model(a2, b2, z_back, t_b)
    loss_back = torch.mean(((pred_back - T_INITIAL) / T_SCALE)**2)
    
    # ---- FACES 3 & 4: Side walls x=0 and x=L — Insulated (dT/dx = 0) ----
    a3, b3, t3 = sample_face(N_BC)
    x_left = torch.zeros(N_BC, 1, device=device, requires_grad=True)
    pred_xl = model(x_left, a3, b3, t3)
    dT_dx_left = torch.autograd.grad(pred_xl, x_left, torch.ones_like(pred_xl), create_graph=True)[0]
    loss_x0 = torch.mean((dT_dx_left / T_SCALE)**2)
    
    a4, b4, t4 = sample_face(N_BC)
    x_right = (torch.ones(N_BC, 1, device=device) * L).requires_grad_(True)
    pred_xr = model(x_right, a4, b4, t4)
    dT_dx_right = torch.autograd.grad(pred_xr, x_right, torch.ones_like(pred_xr), create_graph=True)[0]
    loss_xL = torch.mean((dT_dx_right / T_SCALE)**2)
    
    # ---- FACES 5 & 6: Side walls y=0 and y=L — Insulated (dT/dy = 0) ----
    a5, b5, t5 = sample_face(N_BC)
    y_bottom = torch.zeros(N_BC, 1, device=device, requires_grad=True)
    pred_yb = model(a5, y_bottom, b5, t5)
    dT_dy_bottom = torch.autograd.grad(pred_yb, y_bottom, torch.ones_like(pred_yb), create_graph=True)[0]
    loss_y0 = torch.mean((dT_dy_bottom / T_SCALE)**2)
    
    a6, b6, t6 = sample_face(N_BC)
    y_top = (torch.ones(N_BC, 1, device=device) * L).requires_grad_(True)
    pred_yt = model(a6, y_top, b6, t6)
    dT_dy_top = torch.autograd.grad(pred_yt, y_top, torch.ones_like(pred_yt), create_graph=True)[0]
    loss_yL = torch.mean((dT_dy_top / T_SCALE)**2)
    
    # ---- Thermodynamic Floor (T >= T_INITIAL everywhere) ----
    x_fl, y_fl, z_fl, t_fl = sample_domain(N_PDE)
    T_domain = model(x_fl, y_fl, z_fl, t_fl)
    loss_floor = torch.mean(torch.relu(T_INITIAL - T_domain)**2)
    
    # ---- Sum of all 6 boundary losses ----
    loss_bc_total = loss_front + loss_back + loss_x0 + loss_xL + loss_y0 + loss_yL
    
    return loss_pde, loss_bc_total, loss_floor

# ===================== PHASE 1: ADAM =====================
print("Phase 1: Adam optimizer (3D)...")
optimizer_adam = optim.Adam(model.parameters(), lr=2e-3)
history = {'total': [], 'pde': [], 'bc': [], 'floor': []}

for epoch in range(ADAM_EPOCHS):
    optimizer_adam.zero_grad()
    lp, lbc, lf = compute_all_losses(model)
    total = 1.0 * lp + 1000.0 * lbc + 100.0 * lf
    total.backward()
    optimizer_adam.step()
    
    history['total'].append(total.item())
    history['pde'].append(lp.item())
    history['bc'].append(lbc.item())
    history['floor'].append(lf.item())
    if epoch % 100 == 0:
        print(f"Epoch {epoch} | Total Loss: {total.item():.4f}") # Optional: Keeps your screen clean
        torch.save(model.state_dict(), 'met_shield_3d_checkpoint.pth')

print("Phase 1 complete.\n")

# ===================== PHASE 2: L-BFGS =====================
print("Phase 2: L-BFGS optimizer (3D)...")
optimizer_lbfgs = optim.LBFGS(model.parameters(), lr=1, max_iter=LBFGS_MAX_ITER, line_search_fn="strong_wolfe")

def closure():
    optimizer_lbfgs.zero_grad()
    lp, lbc, lf = compute_all_losses(model)
    total = 1.0 * lp + 2000.0 * lbc + 100.0 * lf
    total.backward()
    return total

optimizer_lbfgs.step(closure)
if epoch % 100 == 0:
        print(f"Epoch {epoch} | Total Loss: {total.item():.4f}") # Optional: Keeps your screen clean
        torch.save(model.state_dict(), 'met_shield_3d_checkpoint.pth')
print("Phase 2 complete.\n")

# ==========================================
# SAVE FINAL WEIGHTS & GRADIENTS
# ==========================================

# 1. Save trained model weights
weights_path = 'met_shield_3d_weights.pth'
torch.save(model.state_dict(), weights_path)
print(f"Model weights saved to: {weights_path}")

# 2. Run one final forward+backward pass to populate gradients
optimizer_adam.zero_grad()
lp, lbc, lf = compute_all_losses(model)
total = 1.0 * lp + 2000.0 * lbc + 100.0 * lf
total.backward()

# 3. Extract and save final gradients
final_gradients = {}
for name, param in model.named_parameters():
    if param.grad is not None:
        final_gradients[name] = param.grad.clone()

gradients_path = 'met_shield_3d_final_gradients.pt'
torch.save(final_gradients, gradients_path)
print(f"Final gradients saved to: {gradients_path}")
print(f"  Total parameters with gradients: {len(final_gradients)}")
print(f"  Final total loss: {total.item():.6f}")


Phase 1: Adam optimizer (3D)...
  Epoch     0 | Total: 601547.687500 | PDE: 0.004993 | BC: 1.089390 | Floor: 6004.583008
  Epoch   100 | Total: 10340.283203 | PDE: 27.693533 | BC: 10.312590 | Floor: 0.000000
  Epoch   200 | Total: 1605.388672 | PDE: 2.384531 | BC: 1.603004 | Floor: 0.000000
  Epoch   300 | Total: 553.323914 | PDE: 0.073606 | BC: 0.551587 | Floor: 0.016632
  Epoch   400 | Total: 235.190842 | PDE: 0.189054 | BC: 0.234763 | Floor: 0.002386
  Epoch   500 | Total: 182.659286 | PDE: 0.166398 | BC: 0.182366 | Floor: 0.001269


In [ ]:
from IPython.core import compilerop


pip install numpy pandas geopandas shapely